In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import xgboost as xgb 
from scipy.spatial import cKDTree
import re
import warnings

# Suppress warnings related to label encoder being deprecated in XGBoost 
warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')

print("Libraries imported successfully.")

Libraries imported successfully.


In [6]:
# --- Data Loading ---
try:
    dfa=pd.read_csv("enriched_prepared_obs.csv")
    dfb=pd.read_csv('enriched_prepared_plant2_obs.csv')
    df_pest_raw = pd.read_csv('enriched_prepared_pest_obs.csv')
    print("Data files loaded.")
except FileNotFoundError:
    print("Error: Could not find one of the required CSV files. Ensure files are in the working directory.")
    raise

# 1. Merge plant data
plant = pd.concat([dfa, dfb], axis=0, ignore_index=True)

# 2. Clean dates
def clean_date_column(df):
    df['observed_on'] = df['observed_on'].astype(str)
    df['observed_on'] = df['observed_on'].apply(
        lambda x: re.search(r'\d{4}[-/]\d{2}[-/]\d{2}', x).group(0) if re.search(r'\d{4}[-/]\d{2}[-/]\d{2}', x) else x
    )
    df['observed_on'] = df['observed_on'].str.replace('/', '-', regex=False)
    return df

plant = clean_date_column(plant)
df_pest_raw = clean_date_column(df_pest_raw)

# 3. Create biome_cat
def create_biome_cat(df):
    conditions = [
        (df['biome'].between(1, 5)),
        (df['biome'].between(6, 8)),
        (df['biome'].between(9, 14)),
        (df['biome'].between(15, 21)),
        (df['biome'].between(22, 24)),
        (df['biome'] >= 25)
    ]
    choices = ['A', 'B', 'C', 'D', 'E', 'H']
    df['biome_cat'] = np.select(conditions, choices, default='Unknown')
    return df

plant = create_biome_cat(plant)
df_pest = create_biome_cat(df_pest_raw)

# 4. Filter species/pests with < 600 observations
def filter_min_obs(df, col):
    species_counts = df[col].value_counts()
    valid_species = species_counts[species_counts >= 600].index
    return df[df[col].isin(valid_species)].copy()

plant = filter_min_obs(plant, 'scientific_name')
df_pest = filter_min_obs(df_pest, 'scientific_name')

# 5. Map nearest pest to plant data
def map_nearest_pest_no_nan(plant_df, pest_df):
    merged_list = []
    for biome in plant_df['biome_cat'].unique():
        plant_b = plant_df[plant_df['biome_cat']==biome].copy()
        pest_b = pest_df[pest_df['biome_cat']==biome].copy()
        if pest_b.empty:
            pest_b = pest_df.copy()
            
        tree = cKDTree(pest_b[['T2M','PRECTOTCORR','SWGDN']].values)
        _, idxs = tree.query(plant_b[['T2M','PRECTOTCORR','SWGDN']].values)

        plant_b['scientific_name_pest'] = pest_b.iloc[idxs]['scientific_name'].values
        plant_b['common_name_pest'] = pest_b.iloc[idxs]['common_name'].values
        
        merged_list.append(plant_b)
    return pd.concat(merged_list, ignore_index=True)

merged_full = map_nearest_pest_no_nan(plant, df_pest)
print(f"Merged dataset created with {len(merged_full)} observations.")

# 6. Drop unneeded columns and add feature engineering
merged_full.drop(columns=['id','latitude','longitude','taxon_id','biome'],inplace=True)

# Feature Engineering: Cyclic encoding for DOY
merged_full['doy_sin'] = np.sin(2 * np.pi * merged_full['doy'] / 365)
merged_full['doy_cos'] = np.cos(2 * np.pi * merged_full['doy'] / 365)

Data files loaded.
Merged dataset created with 653789 observations.


In [7]:
# Encode categorical features
enc_biome = LabelEncoder()
merged_full['biome_enc'] = enc_biome.fit_transform(merged_full['biome_cat'])

# Encode all target columns and species feature
y_cols = ['scientific_name', 'common_name', 'scientific_name_pest', 'common_name_pest', 'phenophase']
label_encoders = {}
for col in y_cols:
    le = LabelEncoder()
    merged_full[col + '_enc'] = le.fit_transform(merged_full[col])
    label_encoders[col] = le

# Encode scientific_name for phenophase feature
enc_species = LabelEncoder()
merged_full['species_enc'] = enc_species.fit_transform(merged_full['scientific_name'])

# Input features for the model training
model_features = ['biome_enc', 'T2M', 'PRECTOTCORR', 'SWGDN', 'doy_sin', 'doy_cos']

print("Data encoding complete.")

Data encoding complete.


In [8]:
# --- Check for GPU and Set Parameters ---
try:
    # Check if a GPU is available for XGBoost
    if xgb.get_config()["get_gpu_support"]:
        T_METHOD = 'gpu_hist'
        PREDICTOR = 'gpu_predictor'
        print("✅ GPU support detected. Training will use 'gpu_hist' for acceleration.")
    else:
        T_METHOD = 'hist'
        PREDICTOR = 'auto'
        print("⚠️ GPU support not detected or configured. Falling back to CPU 'hist' method.")
except:
    T_METHOD = 'hist'
    PREDICTOR = 'auto'
    print("⚠️ XGBoost configuration check failed. Falling back to CPU 'hist' method.")


models = {}

# GPU/Optimized Parameters
xgb_params = {
    'objective': 'multi:softprob',
    'eval_metric': 'mlogloss',
    'tree_method': T_METHOD,      # Dynamic setting (gpu_hist or hist)
    'n_estimators': 300,            
    'max_depth': 12,                
    'learning_rate': 0.05,          
    'use_label_encoder': False,
    'random_state': 42,
    'n_jobs': -1,
    'predictor': PREDICTOR         # Dynamic setting (gpu_predictor or auto)
}

# 1️⃣ Plant Species Prediction Model
target_species = 'scientific_name_enc'
X_train, _, y_train, _ = train_test_split(
    merged_full[model_features], merged_full[target_species], test_size=0.2, random_state=42
)
xgb_species = xgb.XGBClassifier(**xgb_params, num_class=len(label_encoders['scientific_name'].classes_))
print("Starting Species Model Training...")
xgb_species.fit(X_train, y_train)
models['scientific_name'] = xgb_species
print("Species Model Trained.")


# 2️⃣ Phenophase Prediction Model
X_phase = merged_full[model_features + ['species_enc']]
y_phase = merged_full['phenophase_enc']
X_train, _, y_train, _ = train_test_split(X_phase, y_phase, test_size=0.2, random_state=42)
xgb_phase = xgb.XGBClassifier(**xgb_params, num_class=len(label_encoders['phenophase'].classes_))
print("Starting Phenophase Model Training...")
xgb_phase.fit(X_train, y_train)
models['phenophase'] = xgb_phase
print("Phenophase Model Trained.")


# 3️⃣ Pest Prediction Model
target_pest = 'scientific_name_pest_enc'
X_train, _, y_train, _ = train_test_split(
    merged_full[model_features], merged_full[target_pest], test_size=0.2, random_state=42
)
xgb_pest = xgb.XGBClassifier(**xgb_params, num_class=len(label_encoders['scientific_name_pest'].classes_))
print("Starting Pest Model Training...")
xgb_pest.fit(X_train, y_train)
models['scientific_name_pest'] = xgb_pest
print("All Models Trained. Ready for Prediction.")

⚠️ XGBoost configuration check failed. Falling back to CPU 'hist' method.
Starting Species Model Training...


KeyboardInterrupt: 

In [ ]:
import os
import sys

print("--- System Python Path ---")
print(sys.executable)

# 1. Uninstall existing XGBoost
!pip uninstall -y xgboost

# 2. Reinstall XGBoost
# The installation should now use your correctly set environment paths after the reboot.
print("\n--- Reinstalling XGBoost ---")
!pip install xgboost

# 3. Restart the kernel to load the newly installed GPU-aware module
print("\nInstallation finished. Please manually RESTART YOUR KERNEL now.")

--- System Python Path ---
C:\Users\aayus\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe
Found existing installation: xgboost 3.0.5
Uninstalling xgboost-3.0.5:
  Successfully uninstalled xgboost-3.0.5

--- Reinstalling XGBoost ---


You can safely remove it manually.


Defaulting to user installation because normal site-packages is not writeable
  Using cached xgboost-3.0.5-py3-none-win_amd64.whl.metadata (2.1 kB)
Using cached xgboost-3.0.5-py3-none-win_amd64.whl (56.8 MB)

Installation finished. Please manually RESTART YOUR KERNEL now.


--- XGBoost GPU Configuration Check ---
❌ FINAL ERROR: 'get_gpu_support'

Training will proceed on CPU, as a hard error is preventing GPU linking.
If you still want GPU acceleration, you must use a Conda environment.


In [ ]:
import sys
print(sys.executable)

d:\Programing\Machine learning\venv\Scripts\python.exe


In [ ]:
!pip install xgboost